# `groundinsight` — plotting helpers

Small visual demo of the three matplotlib helpers shipped with the
package:

- `plot_bus_voltages(result, frequencies=[...])`
- `plot_bus_currents(result, frequencies=[...])`
- `plot_branch_currents(result)`

The aim of this notebook is *not* a numerical plausibility check,
but a quick check that the helpers run end-to-end on a slightly
larger network and produce sensible-looking bar plots.


In [ ]:
import sys
import os
# Make the src/ tree importable when running from the notebooks/ folder
project_root = os.path.abspath(os.path.join(os.getcwd(), '..', 'src'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

import numpy as np
import polars as pl
import matplotlib.pyplot as plt

import groundinsight as gi
from groundinsight.models.core_models import BusType, BranchType, ComplexNumber

print('groundinsight', gi.__version__)


In [ ]:
def make_bus_type():
    """Unit bus impedance so the shield path dominates."""
    return BusType(
        name='BusUnit',
        description='Unit-like bus impedance for plausibility tests',
        system_type='Grounded',
        voltage_level=20.0,
        impedance_formula='rho * 0 + 1.0 + I * f * 0',
    )

def make_ms_cable():
    """MV cable with the reference impedances above."""
    return BranchType(
        name='MSCable',
        description='MV cable reference branch',
        grounding_conductor=True,
        self_impedance_formula='(rho * 0 + 0.25 + I * 0.6)*l',
        mutual_impedance_formula='(rho * 0 + 0.0 + I * 0.6)*l',
    )

def make_ohl():
    """Overhead line without shield."""
    return BranchType(
        name='OHLine',
        description='Overhead line without shield',
        grounding_conductor=False,
        self_impedance_formula='NaN',
        mutual_impedance_formula='NaN',
    )


## 1. Build a 6-bus line

In [ ]:
def build_demo():
    net = gi.create_network(name='PlottingDemo',
                            frequencies=[50, 250, 350])
    bus_type = make_bus_type()
    cable = make_ms_cable()
    n = 6
    for i in range(1, n + 1):
        gi.create_bus(name=f'bus{i}', type=bus_type, network=net)
    for i in range(1, n):
        gi.create_branch(name=f'b{i}{i+1}', type=cable,
                         from_bus=f'bus{i}', to_bus=f'bus{i+1}',
                         length=1.0, network=net)
    gi.create_source(name='src', bus='bus1',
                     values={50: 100.0, 250: 100.0, 350: 100.0}, network=net)
    gi.create_fault(name='fault', bus=f'bus{n}',
                    scalings={50: 1.0, 250: 1.0, 350: 1.0}, network=net)
    return net

net = build_demo()
gi.run_fault(net, fault_name='fault')
result = net.results['fault']
print(net)


## 2. EPR per bus

`plot_bus_voltages` shows the EPR of every bus, one bar group per
selected frequency. The fault bus (`bus6`) is the highest by
construction.


In [ ]:
gi.plot_bus_voltages(result=result, frequencies=[50, 250, 350])
plt.show()


## 3. Bus current per bus

`plot_bus_currents` shows the current flowing into the bus's
grounding admittance per bus.


In [ ]:
gi.plot_bus_currents(result=result, frequencies=[50, 250, 350])
plt.show()


## 4. Branch currents

`plot_branch_currents` shows the per-branch current. For a simple
line all branches carry the same current; for a ring/mesh the
parallel-path split becomes visible.


In [ ]:
gi.plot_branch_currents(result=result)
plt.show()


## 5. Single-frequency view

In [ ]:
gi.plot_bus_voltages(result=result, frequencies=[50])
plt.show()


---

**Notes**

- The plot helpers all return a `matplotlib.figure.Figure` and call
  `plt.show()` internally when `show=True` (default).
- For paper-ready output, set figure size and DPI before calling the
  helpers, e.g.:
  ```python
  import matplotlib.pyplot as plt
  plt.rcParams.update({'figure.dpi': 300, 'font.size': 10})
  gi.plot_bus_voltages(result=result, frequencies=[50],
                       show=False, figsize=(3.5, 3.5))
  ```
